# Aprendizado de Máquina — Aula prática 09

## Máquinas de Vetores de Suporte

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Todos os classificadores das Aulas 07 e 08 partiam de uma probabilidade: modelavam
$P(Y=1\mid x)$ e cortavam. A SVM parte de outro lugar, puramente geométrico:

> **entre todas as fronteiras que separam as classes, escolha a que fica mais
> longe dos pontos.**

Dessa ideia saem três coisas que valem o notebook inteiro: a fronteira depende
apenas de **alguns** pontos (os vetores de suporte); a fronteira pode virar curva
sem que ninguém calcule coordenada nova (o truque do *kernel*); e o método não
entrega probabilidade nenhuma — o que, depois da Aula 08, você já sabe que é uma
limitação séria e não um detalhe.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- identificar os vetores de suporte e **verificar** que só eles determinam a
  fronteira;
- explicar o que o $C$ controla e o que acontece com o número de vetores de suporte;
- **conferir numericamente o truque do kernel**, comparando o kernel polinomial com
  a expansão explícita de atributos;
- buscar $C$ e $\gamma$ em grade bidimensional e enxergar por que ela precisa ser
  bidimensional;
- comparar as perdas *hinge*, logística e 0–1;
- medir o custo computacional do `SVC` e confirmar por que ele não escala.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos são o `SVC` e o `LinearSVC`, mais três geradores de dados
artificiais do `sklearn.datasets` — é com eles que as fronteiras ficam visíveis.

In [ ]:
import sklearn.model_selection as skm
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVC, LinearSVC

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def desenhar_pontos(ax, X, y, titulo="", apagados=None):
    if apagados is None:
        apagados = np.zeros(len(y), dtype=bool)
    for classe, cor in [(0, "steelblue"), (1, "crimson")]:
        for selecao, alfa in [(apagados, 0.25), (~apagados, 1.0)]:
            pontos = X[(y == classe) & selecao]
            ax.scatter(pontos[:, 0], pontos[:, 1], s=18, color=cor, edgecolor="k", lw=0.3,
                       alpha=alfa)
    ax.set_xlim(X[:, 0].min() - 0.7, X[:, 0].max() + 0.7)
    ax.set_ylim(X[:, 1].min() - 0.7, X[:, 1].max() + 0.7)
    ax.set_title(titulo, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])


def desenhar_fronteira(ax, modelo, X, y, titulo="", margens=False, suporte=False,
                       apagar=False):
    passo = 0.02
    x0 = np.arange(X[:, 0].min() - 0.7, X[:, 0].max() + 0.7, passo)
    x1 = np.arange(X[:, 1].min() - 0.7, X[:, 1].max() + 0.7, passo)
    gx, gy = np.meshgrid(x0, x1)
    Z = modelo.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, Z > 0, levels=[-0.5, 0.5, 1.5],
                colors=["steelblue", "crimson"], alpha=0.10)
    niveis = [-1, 0, 1] if margens else [0]
    estilos = ["--", "-", "--"] if margens else ["-"]
    ax.contour(gx, gy, Z, levels=niveis, colors="black", linestyles=estilos,
               linewidths=[1, 1.6, 1] if margens else [1.6])
    if suporte or apagar:
        svc = modelo if hasattr(modelo, "support_") else modelo[-1]
    apagados = None
    if apagar:                  # apaga as observacoes que nao sao vetores de suporte
        apagados = np.ones(len(y), dtype=bool)
        apagados[svc.support_] = False
    desenhar_pontos(ax, X, y, titulo, apagados)
    if suporte:                 # circula os vetores de suporte
        sv = svc.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=110, facecolors="none", edgecolors="green",
                   linewidths=1.3)

---
## 2. A margem, e os pontos que a seguram

Começamos pelo caso mais simples, o do classificador de margem máxima, em que uma reta
separa as duas classes. Os dados são 60 observações em $\mathbb{R}^2$, 30 de cada
classe, geradas pelo `make_blobs` em torno de dois centros que a própria função sorteia:
cada observação é o centro da sua classe somado a um vetor normal de coordenadas
independentes, com média zero e desvio-padrão 1,05. Antes de ajustar qualquer coisa,
olhamos para os dados. Na figura, o eixo horizontal é $x_1$ e o vertical é $x_2$; a
classe 0 está em azul, e a classe 1, em vermelho.

In [ ]:
X, y = make_blobs(n_samples=60, centers=2, cluster_std=1.05, random_state=6)

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_pontos(ax, X, y, "as 60 observacoes")

print(f"observacoes: {len(y)}  (classe 0: {np.sum(y == 0)}, classe 1: {np.sum(y == 1)})")

As duas nuvens não se misturam: há uma faixa vazia entre elas, e muitas retas deixam a
classe 0 inteira de um lado e a classe 1 inteira do outro. Para tornar isso preciso,
seguimos a notação das notas. Codificamos as classes por $y_i \in \{-1, +1\}$, com $-1$
para a classe 0 e $+1$ para a classe 1, e escrevemos cada reta como o conjunto dos
pontos em que se anula a função

$$f(\mathbf{x}) = \beta_0 + \beta_1 x_1 + \beta_2 x_2 = \beta_0 + \boldsymbol{\beta}^\top \mathbf{x},
\qquad \boldsymbol{\beta} = (\beta_1, \beta_2)^\top \neq \mathbf{0}.$$

A reta é **separadora** quando cada observação está do lado em que $f$ tem o sinal da
sua classe, isto é, quando $y_i f(\mathbf{x}_i) > 0$ para $i = 1, \dots, n$. Como se
mostra nas notas, a distância de um ponto $\mathbf{x}$ à reta é
$|f(\mathbf{x})|/\|\boldsymbol{\beta}\|$, e, se a reta é separadora,
$|f(\mathbf{x}_i)| = y_i f(\mathbf{x}_i)$ para toda observação. A menor distância de uma
observação à reta é a **margem** da reta:

$$m(\beta_0, \boldsymbol{\beta}) = \min_{1 \le i \le n} \frac{y_i f(\mathbf{x}_i)}{\|\boldsymbol{\beta}\|}.$$

As notas definem a margem com $\|\boldsymbol{\beta}\| = 1$; dividir por
$\|\boldsymbol{\beta}\|$ dá o mesmo número sem normalizar os coeficientes.

A célula a seguir desenha três retas que escolhemos olhando a figura anterior, com
inclinações diferentes, e confere para cada uma se ela é separadora e qual é a sua
margem, calculada pela função `margem`. Cada reta é dada na forma $x_2 = a + b\,x_1$,
que é a reta $f(\mathbf{x}) = 0$ com $\beta_0 = a$, $\beta_1 = b$ e $\beta_2 = -1$. Com
esse sinal de $\beta_2$, $f$ é positiva abaixo da reta, que é onde está a classe $+1$.

In [ ]:
y_pm = 2 * y - 1        # classe 0 -> -1 e classe 1 -> +1


def margem(beta0, beta, X, y_pm):
    # menor valor de y_i f(x_i) / ||beta||, que e' a margem quando a reta separa as classes
    return np.min(y_pm * (beta0 + X @ beta)) / np.linalg.norm(beta)


retas = {"reta 1": (-1.5, -0.6), "reta 2": (-8.0, 0.25), "reta 3": (-6.9, 0.0)}   # (a, b)

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_pontos(ax, X, y, "tres retas separadoras")
extremos = np.array(ax.get_xlim())          # as duas pontas do eixo horizontal
for (nome, (a, b)), cor in zip(retas.items(), ["darkorange", "purple", "dimgray"]):
    ax.plot(extremos, a + b * extremos, color=cor, lw=1.6, label=nome)
ax.legend(fontsize=8, loc="upper right")

for nome, (a, b) in retas.items():
    beta0, beta = a, np.array([b, -1.0])
    separa = np.all(y_pm * (beta0 + X @ beta) > 0)
    print(f"{nome}: x2 = {a:.2f} {b:+.2f} x1   separa as classes? {separa}   "
          f"margem {margem(beta0, beta, X, y_pm):.4f}")

As três retas são separadoras: qualquer uma delas classifica corretamente as 60
observações de treino. As margens, porém, são diferentes: 0,5468 na reta 1, 0,7661 na
reta 2 e 0,4291 na reta 3, e cada uma é a distância da reta à observação mais próxima
dela. E essas três não são as únicas. As desigualdades $y_i f(\mathbf{x}_i) > 0$ são
estritas e em número finito, e por isso continuam valendo depois de qualquer mudança
suficientemente pequena de $(\beta_0, \boldsymbol{\beta})$: há infinitas retas
separadoras. É preciso um critério para escolher uma delas, e o das notas é ficar o mais
longe possível das observações de treino. O **classificador de margem máxima** é a reta
separadora de maior margem, isto é, a solução do problema

$$\max_{\beta_0,\, \boldsymbol{\beta},\, M} \ M \quad \text{sujeito a} \quad
\sum_{j=1}^{p} \beta_j^2 = 1 \quad \text{e} \quad
y_i\, (\beta_0 + \boldsymbol{\beta}^\top \mathbf{x}_i) \ge M, \ \ i = 1, \dots, n,$$

aqui com $p = 2$.

O `scikit-learn` não tem uma classe só para esse problema. O `SVC` com
`kernel="linear"` resolve um problema mais geral, o do classificador de vetor de
suporte, que é o assunto da Seção 3. Para estes dados, com `C=1000`, a solução que ele
devolve é a do classificador de margem máxima; o que o argumento `C` controla fica para
a Seção 3. A célula ajusta o `SVC` e calcula a margem da reta obtida com a mesma função
`margem`. Na figura, a reta cheia é o classificador, as tracejadas são as paralelas a
ele à distância $M$, o fundo mostra a classe que ele atribui a cada ponto do plano, e os
pontos circulados em verde são os **vetores de suporte**: as observações que estão à
distância $M$ do classificador.

In [ ]:
svm = SVC(kernel="linear", C=1000).fit(X, y)
beta0, beta = svm.intercept_[0], svm.coef_.ravel()

# distancia de cada observacao a reta (todas estao do lado certo dela)
distancia = y_pm * (beta0 + X @ beta) / np.linalg.norm(beta)
outras = np.setdiff1d(np.arange(len(y)), svm.support_)

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_fronteira(ax, svm, X, y, "classificador de margem maxima", margens=True, suporte=True)

print(f"reta: x2 = {-beta0 / beta[1]:.2f} {-beta[0] / beta[1]:+.2f} x1")
print(f"margem M = {margem(beta0, beta, X, y_pm):.4f}")
print(f"vetores de suporte: {svm.n_support_.sum()}, de indices {svm.support_}  (circulados em verde)")
print(f"   distancias a reta: {distancia[svm.support_].round(4)}")
print(f"menor distancia entre as outras {len(outras)} observacoes: {distancia[outras].min():.4f}")

A reta de margem máxima é $x_2 = -5{,}98 - 0{,}02\,x_1$, quase horizontal, e a sua
margem é $M = 1{,}1991$, maior que a de cada uma das três retas anteriores. Os vetores
de suporte são três: as observações 0 e 53, da classe $-1$, e a 49, da classe $+1$. As
distâncias delas à reta coincidem com $M$ até a terceira casa decimal; a diferença, de
alguns décimos de milésimo, vem de o `SVC` resolver o problema numericamente, com
tolerância de $10^{-3}$. A observação seguinte mais próxima já está a $1{,}3201$, e
nenhuma observação fica entre as tracejadas, como exigem as restrições do problema.

As notas afirmam que o classificador de margem máxima depende apenas dos vetores de
suporte: retirar uma observação que não é vetor de suporte não altera a solução. A
célula seguinte confere isso de duas maneiras. Primeiro, ajusta o classificador de novo
só com os três vetores de suporte, descartando as outras 57 observações; depois, com
todas as observações menos um dos vetores de suporte. Em cada caso, imprime a maior
diferença entre os coeficientes obtidos e os originais.

In [ ]:
coef_original = np.r_[svm.coef_.ravel(), svm.intercept_]

# (a) remover TODOS os pontos que nao sao vetores de suporte
mantidos = np.zeros(len(y), dtype=bool)
mantidos[svm.support_] = True
svm_so_sv = SVC(kernel="linear", C=1000).fit(X[mantidos], y[mantidos])
d_a = np.abs(np.r_[svm_so_sv.coef_.ravel(), svm_so_sv.intercept_] - coef_original).max()

# (b) remover UM vetor de suporte
fora = np.ones(len(y), dtype=bool)
fora[svm.support_[0]] = False
svm_sem_um = SVC(kernel="linear", C=1000).fit(X[fora], y[fora])
d_b = np.abs(np.r_[svm_sem_um.coef_.ravel(), svm_sem_um.intercept_] - coef_original).max()

print(f"jogando fora as {int((~mantidos).sum())} observacoes que NAO sao vetores de suporte:")
print(f"   maior mudanca nos coeficientes: {d_a:.2e}")
print(f"jogando fora UM vetor de suporte:")
print(f"   maior mudanca nos coeficientes: {d_b:.4f}")

Descartar 57 das 60 observações mexe na fronteira na **terceira casa decimal** — e
o que sobra é tolerância do otimizador, não estrutura. Descartar um único dos três
vetores de suporte muda mais de cem vezes isso.

Isso separa a SVM de quase todo o resto do curso. A regressão logística usa cada
observação — mover um ponto longe da fronteira muda (pouco, mas muda) os
coeficientes. A SVM ignora completamente quem já está do lado certo e fora da faixa.
É uma virtude, porque dá robustez a pontos distantes, e é um risco, porque toda a
decisão fica nas mãos de pouquíssimas observações — inclusive de eventuais erros de
rotulagem bem na fronteira.

---
## 3. O orçamento $C$, quando as classes se misturam

O classificador de margem máxima só existe quando alguma reta separa as classes, e
dados reais raramente são assim. Nesta seção, as classes se misturam. Os dados são 40
observações, 20 de cada classe, geradas de duas normais bivariadas com coordenadas
independentes de desvio-padrão 0,85, uma centrada em $(-1;\,-0{,}5)$, para a classe 0, e
a outra em $(1;\,0{,}5)$, para a classe 1. Como na seção anterior, começamos olhando para
os dados.

In [ ]:
rng = np.random.default_rng(43)
Xm = np.vstack([rng.normal([-1.0, -0.5], 0.85, size=(20, 2)),
                rng.normal([1.0, 0.5], 0.85, size=(20, 2))])
ym = np.r_[np.zeros(20, dtype=int), np.ones(20, dtype=int)]

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_pontos(ax, Xm, ym, "as 40 observacoes")

print(f"observacoes: {len(ym)}  (classe 0: {np.sum(ym == 0)}, classe 1: {np.sum(ym == 1)})")

Desta vez as nuvens se tocam, e uma observação vermelha aparece no meio das azuis. Isso
sugere que nenhuma reta separa as classes, e o argumento a seguir prova isso sem depender
da impressão visual.

Cada lado de uma reta é um conjunto convexo: se dois pontos estão do mesmo lado, o
segmento entre eles também está. Por isso, se três pontos estão do mesmo lado, o
triângulo que eles formam fica inteiro desse lado. Uma reta que deixasse a classe 0
inteira de um lado deixaria desse lado, então, todo triângulo com vértices em
observações da classe 0, e também qualquer observação que estivesse dentro de um desses
triângulos. Basta, portanto, encontrar uma observação da classe 1 no interior de um
triângulo de observações da classe 0.

A célula a seguir faz essa busca. Um ponto $\mathbf{p}$ está no interior do triângulo de
vértices $\mathbf{a}$, $\mathbf{b}$ e $\mathbf{c}$ quando, percorrendo as arestas no
sentido $\mathbf{a} \to \mathbf{b} \to \mathbf{c} \to \mathbf{a}$, ele fica sempre do
mesmo lado de cada uma; o lado de $\mathbf{p}$ em relação à reta orientada de
$\mathbf{u}$ para $\mathbf{v}$ é dado pelo sinal de
$(v_1 - u_1)(p_2 - u_2) - (v_2 - u_2)(p_1 - u_1)$. A função `dentro` faz esse teste, e a
função `procurar_triangulo` percorre as observações da classe 1 e, para cada uma, os
triângulos de observações da classe 0, até achar um que a contenha.

In [ ]:
from itertools import combinations


def dentro(p, a, b, c):
    # p esta no interior do triangulo abc se fica do mesmo lado das arestas ab, bc e ca
    def lado(u, v):
        return np.sign((v[0] - u[0]) * (p[1] - u[1]) - (v[1] - u[1]) * (p[0] - u[0]))
    return lado(a, b) == lado(b, c) == lado(c, a) != 0


def procurar_triangulo(X, y):
    # uma observacao da classe 1 no interior de um triangulo de observacoes da classe 0
    for i in np.flatnonzero(y == 1):
        for t in combinations(np.flatnonzero(y == 0), 3):
            if dentro(X[i], *X[list(t)]):
                return int(i), [int(k) for k in t]


intrusa, triangulo = procurar_triangulo(Xm, ym)
print(f"a observacao {intrusa}, da classe 1, esta no interior do triangulo "
      f"das observacoes {triangulo}, da classe 0")

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_pontos(ax, Xm, ym, f"a observacao {intrusa} num triangulo da classe 0")
ax.fill(Xm[triangulo, 0], Xm[triangulo, 1], color="steelblue", alpha=0.2)
for k in [intrusa] + triangulo:
    ax.annotate(str(k), Xm[k], xytext=(4, 4), textcoords="offset points", fontsize=8)

A observação 39, da classe 1, está no interior do triângulo das observações 0, 1 e 2, da
classe 0, e por isso nenhuma reta separa as classes. O problema do classificador de
margem máxima não tem solução com $M > 0$, e o critério da Seção 2 não se aplica.

O classificador de vetor de suporte, das notas, contorna isso permitindo que algumas
observações violem a margem. A cada observação associa-se uma **folga**
$\varepsilon_i \ge 0$, e o problema passa a ser

$$\begin{aligned}
\max_{\beta_0,\, \boldsymbol{\beta},\, \boldsymbol{\varepsilon},\, M}\ \ & M \\
\text{sujeito a}\ \ & \textstyle\sum_{j=1}^{p} \beta_j^2 = 1, \\
& y_i\,(\beta_0 + \boldsymbol{\beta}^\top \mathbf{x}_i) \ge M(1 - \varepsilon_i), \quad i = 1, \dots, n, \\
& \varepsilon_i \ge 0, \quad i = 1, \dots, n, \qquad \textstyle\sum_{i=1}^{n} \varepsilon_i \le C,
\end{aligned}$$

em que $C \ge 0$ é o **orçamento** de violações. Numa solução, cada folga vale
$\varepsilon_i = \max\{0,\ 1 - y_i f(\mathbf{x}_i)/M\}$, e o seu valor diz onde está a
observação: $\varepsilon_i = 0$ se ela está do lado certo da margem ou sobre ela;
$0 < \varepsilon_i < 1$ se está dentro da faixa, mas do lado certo do hiperplano;
$\varepsilon_i = 1$ se está sobre o hiperplano; e $\varepsilon_i > 1$ se está do lado
errado do hiperplano, isto é, se é classificada errado. Com $C = 0$, todas as folgas são
nulas, e o problema é o do classificador de margem máxima. Quanto maior o orçamento, mais
violações são toleradas, e a margem $M$ ótima não diminui. Os **vetores de suporte** são
as observações sobre a margem ou do lado errado dela, isto é, com
$y_i f(\mathbf{x}_i) \le M$.

O `SVC` resolve esse problema, mas não recebe o orçamento. O argumento `C` dele é outro
parâmetro, que age no sentido oposto: quanto maior o argumento, menor, ou igual, o
orçamento. Pode-se mostrar que, para cada valor do argumento, a solução devolvida pelo
`SVC` é a solução do problema acima com orçamento igual à soma das folgas dessa solução.
Para saber com que orçamento estamos trabalhando, então, ajustamos o `SVC` e somamos as
folgas.

Para calcular as folgas, é preciso saber em que escala o `SVC` devolve os coeficientes.
Ele devolve $\mathbf{w}$ (`coef_`) e $b$ (`intercept_`) numa escala em que as tracejadas
são as retas em que $b + \mathbf{w}^\top\mathbf{x}$ vale $-1$ e $+1$. Na normalização das
notas, $\boldsymbol{\beta} = \mathbf{w}/\|\mathbf{w}\|$ e $\beta_0 = b/\|\mathbf{w}\|$; a
distância do hiperplano a cada tracejada é $M = 1/\|\mathbf{w}\|$; e
$y_i f(\mathbf{x}_i)/M = y_i\,(b + \mathbf{w}^\top\mathbf{x}_i)$. A função
`margem_e_folgas` faz essas contas.

A célula a seguir ajusta o `SVC` com o argumento `C=0.5`, imprime as observações de folga
positiva e escreve, ao lado de cada uma, a sua folga. Folgas menores que 0,001 não
contam: são de observações sobre a margem, com folga nula, que o otimizador, de
tolerância $10^{-3}$, deixa um pouco para dentro da faixa.

In [ ]:
ym_pm = 2 * ym - 1        # classe 0 -> -1 e classe 1 -> +1


def margem_e_folgas(modelo, X, y_pm):
    # M, os numeros y_i f(x_i) / M e as folgas das notas, a partir de coef_ e intercept_
    w, b = modelo.coef_.ravel(), modelo.intercept_[0]
    razao = y_pm * (b + X @ w)
    return 1 / np.linalg.norm(w), razao, np.maximum(0, 1 - razao)


modelo = SVC(kernel="linear", C=0.5).fit(Xm, ym)
M, razao, folga = margem_e_folgas(modelo, Xm, ym_pm)
violam = np.flatnonzero(folga > 1e-3)

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_fronteira(ax, modelo, Xm, ym, f"as folgas (soma {folga.sum():.2f})", margens=True)
for k in violam:
    ax.annotate(f"{folga[k]:.2f}", Xm[k], xytext=(4, 4), textcoords="offset points", fontsize=7)

print(f"M = {M:.4f}")
print(f"orcamento = soma das folgas = {folga.sum():.4f}")
print(f"do lado errado do hiperplano (folga > 1): {np.sum(folga > 1)}")
print("vetores de suporte do SVC = observacoes com y f(x)/M <= 1 (a menos de 0,001)? "
      f"{np.array_equal(np.sort(modelo.support_), np.flatnonzero(razao <= 1 + 1e-3))}")
pd.DataFrame({"classe": ym_pm[violam], "y f(x) / M": razao[violam], "folga": folga[violam]},
             index=pd.Index(violam, name="observacao")).sort_values("folga", ascending=False).round(4)

Com o argumento `C=0.5`, a margem é $M = 0{,}6292$ e o orçamento usado é $4{,}8340$, a
soma das oito folgas da tabela. A observação 39 tem folga $2{,}4647 > 1$: está do lado
errado do hiperplano, e é a única observação de treino classificada errado, menos que o
orçamento, como garante a proposição das notas. As outras sete têm folgas entre 0 e 1:
estão dentro da faixa, mas do lado certo do hiperplano. As 32 observações restantes estão
do lado certo da margem e não gastam orçamento algum. Os vetores de suporte do `SVC` são
exatamente as observações com $y_i f(\mathbf{x}_i)/M \le 1$, como na definição das notas:
as oito da tabela e três que estão sobre as tracejadas.

Para ver o papel do orçamento, ajustamos o `SVC` com vários valores do argumento `C` e,
para cada um, calculamos o orçamento, a margem $M$, o número de violações (observações
com folga positiva), o número de erros de treino (observações do lado errado do
hiperplano) e o número de vetores de suporte. A figura mostra três desses ajustes, com os
argumentos 100, 0,5 e 0,05, que correspondem a orçamentos crescentes. Para não carregar a
figura, os vetores de suporte aparecem em cor cheia, e as demais observações, apagadas.

In [ ]:
linhas = []
for c in [1000, 100, 1, 0.5, 0.1, 0.05]:
    modelo = SVC(kernel="linear", C=c).fit(Xm, ym)
    M, _, folga = margem_e_folgas(modelo, Xm, ym_pm)
    linhas.append({"argumento C": c, "orcamento": folga.sum(), "M": M,
                   "violacoes": int(np.sum(folga > 1e-3)),
                   "erros de treino": int(np.sum(folga > 1)),
                   "vetores de suporte": len(modelo.support_)})

fig, axes = subplots(1, 3, figsize=(11, 3.6))
for ax, c in zip(axes, [100, 0.5, 0.05]):
    modelo = SVC(kernel="linear", C=c).fit(Xm, ym)
    M, _, folga = margem_e_folgas(modelo, Xm, ym_pm)
    desenhar_fronteira(ax, modelo, Xm, ym, f"orcamento {folga.sum():.2f},  M = {M:.2f}",
                       margens=True, apagar=True)

pd.DataFrame(linhas).set_index("argumento C").round(4)

Da esquerda para a direita, o orçamento cresce de 3,11 para 4,83 e para 10,82, e a faixa
se alarga: $M$ passa de 0,34 para 0,63 e para 1,42. A faixa mais larga alcança mais
observações: as que violam a margem passam de 2 para 8 e para 19, e os vetores de
suporte, de 5 para 11 e para 22. A tabela mostra a mesma tendência com mais valores do
argumento: quanto menor o argumento, maior o orçamento, e a margem nunca diminui quando o
orçamento aumenta, como as notas demonstram. Em todos os ajustes há uma única observação
do lado errado do hiperplano, menos que o orçamento, como garante a proposição das notas.

Aumentar o argumento de 100 para 1000 praticamente não muda a solução: o orçamento fica
em 3,11, e $M$, em 0,34. O orçamento não poderia chegar a zero: como as classes não são
separáveis, toda reta deixa ao menos uma observação do lado errado, e a folga dessa
observação passa de 1. Por isso o classificador de margem máxima, o caso $C = 0$, não
existe aqui.

O orçamento regula o balanço entre viés e variância. Com orçamento pequeno, a faixa é
estreita, e a solução depende de poucas observações, as mais próximas da fronteira: 5, no
primeiro painel. Mudar uma delas pode mudar bastante a solução: o viés é baixo e a
variância, alta. Com orçamento grande, a solução depende de muitas observações, 22 no
último painel, e muda pouco quando uma delas muda: o viés é mais alto e a variância, mais
baixa. Como os outros parâmetros de ajuste do curso, o orçamento é escolhido por
validação cruzada (Aula 03), e a Seção 5 faz isso por meio do argumento `C`.

---
## 4. O truque do *kernel*, verificado

Uma fronteira reta não serve para tudo. A saída clássica seria criar atributos
novos — $x_1^2$, $x_1x_2$, $x_2^2$ — e traçar uma reta no espaço ampliado, que vira
uma curva no original. O problema é que o número de atributos explode.

O **truque do kernel** evita isso. A SVM só precisa de **produtos internos** entre
observações, e um kernel calcula o produto interno no espaço ampliado **sem nunca
construir as coordenadas**. Para o kernel polinomial de grau 2,

$$K(x, x') = (\gamma\,\langle x, x'\rangle + r)^2 = \langle \phi(x), \phi(x')\rangle,$$

onde $\phi$ é a expansão quadrática. Isso não é uma analogia — é uma igualdade, e
dá para conferir.

In [ ]:
Xc, yc = make_circles(n_samples=300, factor=0.45, noise=0.10, random_state=1)

# (a) kernel polinomial de grau 2, com gamma=1 e coef0=1
kernel = SVC(kernel="poly", degree=2, gamma=1.0, coef0=1.0, C=5).fit(Xc, yc)

# (b) a expansao explicita das mesmas coordenadas, com um kernel LINEAR
#     (x1, x2) -> (1, x1, x2, x1^2, x1 x2, x2^2), com os pesos binomiais certos
def expandir(M):
    x1, x2 = M[:, 0], M[:, 1]
    return np.c_[np.ones(len(M)), np.sqrt(2) * x1, np.sqrt(2) * x2,
                 x1 ** 2, np.sqrt(2) * x1 * x2, x2 ** 2]


explicito = SVC(kernel="linear", C=5).fit(expandir(Xc), yc)

d_kernel = kernel.decision_function(Xc)
d_explic = explicito.decision_function(expandir(Xc))
print(f"predicoes iguais? {np.array_equal(kernel.predict(Xc), explicito.predict(expandir(Xc)))}")
print(f"maior diferenca na funcao de decisao: {np.abs(d_kernel - d_explic).max():.2e}")
print(f"correlacao entre as duas: {np.corrcoef(d_kernel, d_explic)[0,1]:.10f}")

As duas rotas dão a mesma coisa. Em duas dimensões a expansão tem 6 colunas e o
truque parece desnecessário; em 100 dimensões com grau 3 seriam mais de 170 mil
colunas, e com o kernel RBF seriam **infinitas** — é aí que o truque deixa de ser
elegância e vira a única possibilidade.

In [ ]:
fig, axes = subplots(1, 4, figsize=(11, 3.0))
Xl, yl = make_moons(n_samples=300, noise=0.22, random_state=2)
for ax, (nome, m) in zip(axes, [
        ("linear", SVC(kernel="linear", C=1)),
        ("polinomial grau 3", SVC(kernel="poly", degree=3, C=1, gamma="scale")),
        ("RBF (gamma=1)", SVC(kernel="rbf", gamma=1, C=1)),
        ("RBF (gamma=50)", SVC(kernel="rbf", gamma=50, C=1))]):
    m.fit(Xl, yl)
    desenhar_fronteira(ax, m, Xl, yl, f"{nome}\ntreino {m.score(Xl, yl):.3f}")

O último painel é a lição de sempre, com roupa nova: $\gamma$ grande faz o kernel
RBF enxergar cada ponto isoladamente, e a fronteira vira um conjunto de ilhas em
torno das observações de treino. Acurácia perfeita no treino, e nada aprendido.
O $\gamma$ do RBF é um botão de flexibilidade, exatamente como o grau do polinômio
da Aula 01 e o $1/k$ do KNN da Aula 04.

---
## 5. Por que a grade tem de ser bidimensional

$C$ e $\gamma$ **interagem**: um $\gamma$ maior pode ser compensado por um $C$
menor, e vice-versa. Buscar um de cada vez encontra um ponto ruim. Vamos varrer os
dois e olhar a superfície.

In [ ]:
Xg, yg = make_moons(n_samples=400, noise=0.3, random_state=5)
Cs = np.logspace(-2, 3, 12)
gammas = np.logspace(-2, 2, 12)

busca = skm.GridSearchCV(Pipeline([("sc", StandardScaler()), ("svc", SVC())]),
                         {"svc__C": Cs, "svc__gamma": gammas},
                         cv=5, scoring="accuracy", n_jobs=-1).fit(Xg, yg)
grade = -np.array(busca.cv_results_["mean_test_score"]).reshape(len(Cs), len(gammas))

fig, ax = subplots(figsize=(5.2, 4.0))
im = ax.pcolormesh(gammas, Cs, -grade, shading="auto", cmap="viridis")
ax.plot(busca.best_params_["svc__gamma"], busca.best_params_["svc__C"], "r*", ms=16)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("gamma"); ax.set_ylabel("C")
ax.set_title("acuracia de CV (a estrela e' o otimo)", fontsize=9)
fig.colorbar(im, ax=ax)

print(f"melhor: C = {busca.best_params_['svc__C']:.4g}, "
      f"gamma = {busca.best_params_['svc__gamma']:.4g}, "
      f"acuracia {busca.best_score_:.4f}")

A região boa é uma **faixa diagonal**, não uma cruz. Se você fixasse $\gamma$ no
valor padrão e buscasse só o $C$, andaria por uma linha horizontal do mapa — e
poderia atravessá-lo inteiro sem tocar na faixa. É por isso que a grade é
bidimensional.

E note o `StandardScaler` no `Pipeline`: o kernel RBF depende de
$\|x - x'\|^2$, que soma todas as coordenadas. Sem padronizar, a coluna de maior
amplitude decide o valor do kernel sozinha, e o $\gamma$ ótimo passa a depender das
unidades em que os dados foram medidos.

A acurácia e a AUC não perguntam a mesma coisa, então nada obriga a busca a parar
no mesmo $(C,\gamma)$. Refazemos a grade trocando só o `scoring` — e, de passagem,
descobrimos o que o `scikit-learn` usa como escore quando o modelo não tem
`predict_proba`.

In [ ]:
busca_auc = skm.GridSearchCV(Pipeline([("sc", StandardScaler()), ("svc", SVC())]),
                             {"svc__C": Cs, "svc__gamma": gammas},
                             cv=5, scoring="roc_auc", n_jobs=-1).fit(Xg, yg)

for rotulo, b in [("accuracy", busca), ("roc_auc", busca_auc)]:
    print(f"scoring={rotulo:9s} C = {b.best_params_['svc__C']:8.3f}   "
          f"gamma = {b.best_params_['svc__gamma']:7.4f}   escore {b.best_score_:.4f}")

svc_aj = busca_auc.best_estimator_
print(f"\nSVC tem predict_proba? {hasattr(svc_aj.named_steps['svc'], 'predict_proba')}")
print("o que a AUC usa no lugar: decision_function, a distancia com sinal ate a margem")
d = svc_aj.decision_function(Xg)
print(f"   faixa de valores: de {d.min():.2f} a {d.max():.2f}  "
      f"(nao sao probabilidades)")
print(f"   AUC com decision_function: {roc_auc_score(yg, d):.4f}")

**O $\gamma$ não muda; o $C$ sim.** As duas buscas param em $\gamma=0{,}2848$, e o
$C$ vai de $43{,}3$ (acurácia) para $15{,}2$ (AUC).

Faz sentido que seja assim. O $\gamma$ decide a **forma** da fronteira — quão
ondulada ela pode ser —, e as duas métricas concordam sobre a forma certa. O $C$
decide quanto a fronteira pode errar para ficar mais simples, e é aí que as
perguntas se separam: a acurácia só olha de que lado de $0$ caiu cada ponto, então
tolera um modelo confiante e ocasionalmente errado; a AUC olha a **ordem** dos
escores, e prefere um modelo que erre com pouca margem a um que erre com muita. Um
$C$ menor, que é uma margem mais larga, atende melhor à segunda.

Quanto ao escore: o `SVC` **não tem `predict_proba`** por padrão — obtê-lo exigiria
`probability=True`, que roda uma calibração de Platt por validação cruzada interna e
custa caro. Quando falta `predict_proba`, o `scikit-learn` usa a
`decision_function`: a distância com sinal até a margem, que aqui vai de $-6{,}83$ a
$5{,}38$.

E isso basta, porque **a AUC só depende da ordem**. Ela é a probabilidade de um
positivo sorteado receber escore maior que um negativo sorteado; qualquer
transformação monótona do escore — inclusive a que converteria distância em
probabilidade — deixa a AUC intacta. É a mesma propriedade que a Aula 08 explorou
ao calibrar o Bayes ingênuo sem mexer na AUC.

---
## 6. A perda *hinge*, e a probabilidade que não existe

A SVM minimiza a perda ***hinge***, $\max(0, 1 - y f(x))$ com $y \in \{-1, +1\}$.
Compare com a logística, $\log(1 + e^{-y f(x)})$, e com a perda 0–1, que é a que
realmente queremos minimizar e é intratável.

In [ ]:
u = np.linspace(-3, 3, 400)
fig, ax = subplots(figsize=(5.4, 3.2))
ax.plot(u, np.maximum(0, 1 - u), label="hinge (SVM)", lw=1.8)
ax.plot(u, np.log2(1 + np.exp(-u)), label="logistica (em log2)", lw=1.8)
ax.plot(u, (u <= 0).astype(float), label="0-1 (o que queremos)", lw=1.8, ls="--")
ax.axvline(1, ls=":", color="gray")
ax.set_xlabel("margem  y * f(x)"); ax.set_ylabel("perda")
ax.set_ylim(-0.1, 3.2); ax.legend(fontsize=8)

As duas primeiras são aproximações convexas da terceira, e a diferença entre elas é
tudo. A *hinge* **zera** a partir da margem 1: um ponto classificado com folga não
contribui em nada, e é exatamente por isso que a Seção 2 pôde jogar fora 57
observações. A logística nunca zera — ela sempre quer empurrar o ponto para mais
longe, e é essa cauda que sustenta a interpretação probabilística.

Consequência direta: a SVM não estima $P(Y=1\mid x)$. O `scikit-learn` oferece
`probability=True`, que ajusta uma sigmoide sobre a função de decisão por validação
cruzada interna (*Platt scaling*) — mais lento, e um remendo, não uma dedução.
Vamos ver quanto custa e quanto vale.

In [ ]:
import time

Xp, yp = make_moons(n_samples=1500, noise=0.35, random_state=7)
Xp_te, yp_te = make_moons(n_samples=8000, noise=0.35, random_state=8)

modelos = {
    "SVM (probability=False)": SVC(C=1, gamma="scale"),
    "SVM (probability=True)": SVC(C=1, gamma="scale", probability=True, random_state=0),
    "logistica": Pipeline([("poli", PolynomialFeatures(3)),
                           ("sc", StandardScaler()),
                           ("lg", LogisticRegression(max_iter=5000))]),
}
linhas = []
for nome, m in modelos.items():
    t0 = time.perf_counter(); m.fit(Xp, yp); dt = time.perf_counter() - t0
    escore = (m.predict_proba(Xp_te)[:, 1] if hasattr(m, "predict_proba")
              and getattr(m, "probability", True) else m.decision_function(Xp_te))
    linha = {"modelo": nome, "segundos": dt,
             "acuracia": accuracy_score(yp_te, m.predict(Xp_te)),
             "AUC": roc_auc_score(yp_te, escore), "Brier": np.nan}
    if getattr(m, "probability", True) and hasattr(m, "predict_proba"):
        linha["Brier"] = brier_score_loss(yp_te, m.predict_proba(Xp_te)[:, 1])
    linhas.append(linha)
pd.DataFrame(linhas).set_index("modelo").round(4)

A AUC é praticamente a mesma com e sem `probability=True` — o *ordenamento* já
estava lá, na função de decisão. O que muda é o custo e a existência de um número
entre 0 e 1 que se possa multiplicar por dinheiro.

Se você só precisa ranquear, use `decision_function` e economize o tempo. Se
precisa de probabilidade calibrada, a Aula 08 já disse o que fazer: meça o Brier
antes de confiar.

---
## 7. Por que a SVM não escala

O treino do `SVC` resolve um problema quadrático com uma variável por observação, e
a matriz de kernel tem $n^2$ entradas. A teoria diz que o custo fica entre $O(n^2)$
e $O(n^3)$. Vamos medir o expoente.

In [ ]:
tamanhos = np.array([2000, 4000, 8000, 16_000, 32_000])
tempos = {"SVC (kernel RBF)": [], "LinearSVC": [], "logistica": []}
for n in tamanhos:
    Xs, ys = make_moons(n_samples=int(n), noise=0.35, random_state=1)
    Xs = StandardScaler().fit_transform(Xs)
    for nome, m in [("SVC (kernel RBF)", SVC(C=1, gamma="scale")),
                    ("LinearSVC", LinearSVC(C=1, max_iter=20000)),
                    ("logistica", LogisticRegression(max_iter=5000))]:
        t0 = time.perf_counter(); m.fit(Xs, ys); tempos[nome].append(time.perf_counter() - t0)

for nome, ts in tempos.items():
    expoente = np.polyfit(np.log(tamanhos), np.log(np.maximum(ts, 1e-6)), 1)[0]
    print(f"{nome:18s} expoente medido: {expoente:.2f}   "
          f"tempo em n=32000: {ts[-1]*1000:8.1f} ms")

In [ ]:
fig, ax = subplots(figsize=(5.2, 3.2))
for nome, ts in tempos.items():
    ax.plot(tamanhos, ts, "o-", ms=4, label=nome)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("n"); ax.set_ylabel("segundos para ajustar")
ax.legend(fontsize=8)

O expoente medido para o `SVC` fica em torno de **2**, como a teoria prevê,
enquanto o `LinearSVC` e a logística crescem **linearmente**. E a diferença absoluta
é brutal: em $n = 32\,000$ o `SVC` leva mais de mil vezes o tempo do `LinearSVC`.
Extrapolando com expoente 2, um ajuste que leva alguns segundos em $n=32\,000$
levaria **horas** em $n = 10^6$ — e isso para um único ajuste, sem contar a busca de
$(C,\gamma)$, que multiplica tudo pelo tamanho da grade vezes o número de dobras.

As saídas, quando o problema fica grande: `LinearSVC` (que resolve o caso linear
por outro algoritmo), aproximações do kernel como o `Nystroem` — que constrói
explicitamente umas centenas de coordenadas aproximando o RBF, e aí um modelo
linear resolve — ou simplesmente trocar de família.

---
## 8. Caso real: de volta ao câncer de mama

A mesma base da Aula 07, para poder comparar diretamente.

In [ ]:
from sklearn.datasets import load_breast_cancer

dados = load_breast_cancer()
X_tr, X_ts, y_tr, y_ts = skm.train_test_split(dados.data, dados.target,
                                              test_size=0.3, random_state=0,
                                              stratify=dados.target)

svm_busca = skm.GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("svc", SVC())]),
    {"svc__C": np.logspace(-2, 3, 10), "svc__gamma": np.logspace(-4, 1, 10),
     "svc__kernel": ["rbf", "linear"]},
    cv=5, scoring="roc_auc", n_jobs=-1).fit(X_tr, y_tr)

print("melhores parametros:", svm_busca.best_params_)
print(f"AUC de CV no vencedor: {svm_busca.best_score_:.4f}")

In [ ]:
comparacao = {
    "SVM (C e gamma por CV)": svm_busca,
    "logistica (C por CV)": skm.GridSearchCV(
        Pipeline([("sc", StandardScaler()),
                  ("lg", LogisticRegression(max_iter=5000))]),
        {"lg__C": np.logspace(-3, 3, 13)}, cv=5, scoring="roc_auc").fit(X_tr, y_tr),
}
linhas = []
for nome, m in comparacao.items():
    escore = m.decision_function(X_ts)          # serve para a AUC nos dois casos
    linhas.append({"modelo": nome,
                   "acuracia": accuracy_score(y_ts, m.predict(X_ts)),
                   "AUC": roc_auc_score(y_ts, escore)})

sv = svm_busca.best_estimator_.named_steps["svc"]
print(f"kernel escolhido: {svm_busca.best_params_['svc__kernel']}")
print(f"vetores de suporte: {int(sv.n_support_.sum())} de {len(y_tr)} observacoes "
      f"({sv.n_support_.sum()/len(y_tr):.1%})\n")
pd.DataFrame(linhas).set_index("modelo").round(4)

A SVM e a logística chegam essencialmente ao mesmo lugar — o que é comum em
problemas com sinal forte e fronteira quase linear. A diferença que sobra é
prática: a logística entrega probabilidades calibradas de graça e ajusta em
milissegundos; a SVM entrega uma fronteira definida por uma fração das observações
e um hiperparâmetro a mais para buscar.

A escolha, então, não sai do desempenho: sai de o que você vai fazer com a saída.

O número de vetores de suporte é o tamanho do modelo em disco: são eles que
precisam ser guardados para prever. Vamos ver como ele responde ao $C$.

In [ ]:
melhor_C = svm_busca.best_params_["svc__C"]
melhor_gamma = svm_busca.best_params_["svc__gamma"]
melhor_kernel = svm_busca.best_params_["svc__kernel"]

print(f"{'C':>10} {'vetores de suporte':>20} {'% do treino':>13} {'AUC no teste':>14}")
for C in sorted({0.01, melhor_C, 1000.0}):
    m_C = Pipeline([("sc", StandardScaler()),
                    ("svc", SVC(C=C, gamma=melhor_gamma,
                                kernel=melhor_kernel))]).fit(X_tr, y_tr)
    n_sv = int(m_C.named_steps["svc"].n_support_.sum())
    marca = "  <- vencedor da CV" if C == melhor_C else ""
    print(f"{C:10.3g} {n_sv:20d} {100 * n_sv / len(y_tr):12.1f}% "
          f"{roc_auc_score(y_ts, m_C.decision_function(X_ts)):13.4f}{marca}")

| $C$ | vetores de suporte | % do treino | AUC no teste |
| --- | --- | --- | --- |
| $0{,}01$ (vencedor da CV) | 92 | 23,1% | $0{,}9921$ |
| $1000$ | 24 | 6,0% | $0{,}9911$ |

**$C$ pequeno dá mais vetores de suporte, não menos.** São 92 contra 24, quase
quatro vezes mais — e é exatamente o que a teoria manda: $C$ pequeno é penalidade
fraca sobre as folgas, logo margem larga, logo mais pontos dentro ou sobre a
margem. E todo ponto dentro da margem é vetor de suporte.

Num dispositivo com memória apertada, então, a escolha seria **$C=1000$**: 24
vetores em vez de 92, um modelo quatro vezes menor, ao custo de $0{,}001$ de AUC —
que neste conjunto de teste é menos de um caso.

O que se está trocando não é acurácia por espaço; é **robustez** por espaço. A
margem estreita do $C=1000$ apoia a fronteira em 24 observações específicas, e a
Seção 2 mostrou o que acontece quando uma delas sai: mexer num único vetor
de suporte muda os coeficientes. Com 92, nenhum ponto isolado carrega tanto peso. O
modelo pequeno é mais sensível ao conjunto de treino que lhe calhou — e é isso que
você paga, não os $0{,}001$.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| vetores de suporte | §2 | jogar fora 57 de 60 observações mexe 140× menos que tirar 1 vetor de suporte |
| o orçamento $C$ | §3 | orçamento maior $\Rightarrow$ margem mais larga e mais vetores de suporte; o argumento `C` do `SVC` age no sentido oposto |
| truque do kernel | §4 | kernel polinomial $=$ expansão explícita, até $10^{-14}$ |
| $\gamma$ do RBF | §4 | é um botão de flexibilidade: $\gamma$ grande vira ilhas em torno dos dados |
| grade 2D | §5 | a região boa é uma faixa diagonal — buscar um parâmetro de cada vez erra |
| perda *hinge* | §6 | zera a partir da margem 1, e é daí que vem a esparsidade da §2 |
| probabilidades | §6 | `probability=True` é *Platt scaling*, um remendo; a AUC já estava na `decision_function` |
| custo | §7 | expoente medido $\approx 2$; o `LinearSVC` é linear e mil vezes mais rápido em $n=32\,000$ |
| caso real | §8 | empata com a logística — a escolha sai do uso, não do desempenho |

**Leitura recomendada.** [AME] §8.2 (SVM) e §4.6 (RKHS, que é a formalização do
truque do kernel). [ISLP] Capítulo 9 inteiro: §9.1 (o classificador de margem
máxima), §9.2 (margem flexível e o papel do $C$ — note que a convenção de sinal do
$C$ deles é a **oposta** à do `scikit-learn`), §9.3 (kernels) e §9.5 (a relação com
a regressão logística, que é a nossa Seção 6).

**Para praticar.** `Lista teorica 09.pdf` (teórica, com gabarito) e
`Lista prática 09.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 10 fecha a classificação trazendo o KNN e as árvores para cá,
e põe as fronteiras dos classificadores do curso lado a lado nos mesmos dados — que é
a melhor maneira de fixar a intuição de "flexível × rígido" que percorreu o curso.